# 04 v3.3 (Phase 0 Fixed) — Ekstraksi Pairwise Pure-Vision `DINOv3 SAT-493M` / `DINOv2-L`

Notebook ekstraksi **Deep Learning beku (*Frozen Pure-Vision Extractor*)** di **GPU Kaggle** yang sudah menerapkan seluruh **perbaikan Phase 0** dari Implementation Plan:
1. **Pure Asymmetric `pink_cast` Detector:** `max(0, min(R-G, B-G))` dengan skala terpisah (`/ 255.0` pada `selective_color_norm`, `[0, 1]` native di `compute_artifact_and_phys`) — **tidak pernah menghapus warna hijau hutan, biru laut, atau cahaya api (`0.000`)**, sehingga `TINT_NORM_THR = 0.05` aman dipakai.
2. **Guarded Shared Valid-Scale Cropping + Post-Norm Zero Masking:** Menyamakan `side_shared = min(1024, max(side_pre, side_post))` bila `min/max >= 0.50` untuk mencegah mismatch skala `pre`/`post`, plus mengenolkan piksel batas tidak valid **setelah** normalisasi (`chw[:, sq_msk <= 0.5] = 0.0`) agar tidak mencemari global self-attention ViT.
3. **Cloud-Excluded ViT Pooling (`cloud <= 0.50`) + `P10`/`P15`/`frac_chg_abs` Massive-Damage Guard:** `dino_color_post` dan `dino_changed_embed` di-pool di atas patch bebas awan (`fallback all_valid` hanya jika `< 4` patch bersih tersisa), sementara statistik spasial mencatat varian `P10` & `P15` serta lantai `frac_chg_abs` untuk skenario kerusakan `> 85%`.
4. **Dynamic `5-Prefix-Safe` Slicing (`1 CLS + 4 Register` di `DINOv3 SAT-493M`)** + fallback Apache-2.0 `DINOv2-Large` (`img_size=308`).

In [1]:
!pip install -q -U "transformers>=4.56.0" pyarrow accelerate

import gc
import os
import json
import time
import warnings
warnings.filterwarnings("ignore")

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from scipy.ndimage import distance_transform_edt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
import transformers


def get_hf_token():
    tok = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
    if tok:
        return tok.strip()
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF_TOKEN")
        if tok:
            return tok.strip()
    except Exception:
        pass
    return None


HF_TOKEN = get_hf_token()
print(f"torch={torch.__version__} | transformers={transformers.__version__} | timm={timm.__version__}")
if not HF_TOKEN:
    print("[PERINGATAN] Secret HF_TOKEN tidak ditemukan. Jika DINOv3 gated menolak akses, skrip otomatis memakai fallback Apache-2.0 DINOv2-Large.")
else:
    print("[OK] HF_TOKEN terdeteksi.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
torch=2.11.0+cu130 | transformers=5.24.1 | timm=1.0.30
[PERINGATAN] Secret HF_TOKEN tidak ditemukan. Jika DINOv3 gated menolak akses, skrip otomatis memakai fallback Apache-2.0 DINOv2-Large.


In [2]:
# ==================== KONFIGURASI UTAMA (PHASE 0 FIXED) ====================
SEED = 42
BACKBONE_CHOICE = "dinov3_sat_large"   # Ubah ke "dinov2_large" bila panitia belum mengonfirmasi gated weights
IMG_SIZE_V3 = 256                      # Kelipatan 16 -> 16x16 = 256 patch untuk DINOv3 SAT-493M
DINOV2_FALLBACK_IMG_SIZE = 308         # Kelipatan 14 -> 22x22 = 484 patch (hemat runtime T4 dibanding 336)
BATCH_SIZE = 16
NUM_WORKERS = 2
EXTRACT_GRAY3 = True
TINT_NORM_THR = 0.05                   # Aman karena pink_cast sekarang murni asimetris max(0, min(R-G, B-G))
SHARED_CROP_MIN_RATIO = 0.50           # Terapkan side_shared hanya jika min(side_pre, side_post)/max >= 0.50
CLOUD_PATCH_THR = 0.50                 # Maksimal fraksi awan per patch untuk pooling post/changed
# ===========================================================================

np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = "/kaggle/working/v3_dino_features" if os.path.exists("/kaggle/working") else "./v3_dino_features"
os.makedirs(OUT_DIR, exist_ok=True)


def find_task2_paths():
    for sr in [
        "/kaggle/input/competitions/task-2-datavidia-11-0",
        "/kaggle/input/task-2-datavidia-11-0",
        "/kaggle/input",
        ".",
    ]:
        if not os.path.exists(sr):
            continue
        for root, _, files in os.walk(sr):
            if "sample_submission.csv" in files:
                cand_sub = os.path.join(root, "sample_submission.csv")
                probe_id = str(pd.read_csv(cand_sub, nrows=1)["Id"].iloc[0])
                if os.path.exists(os.path.join(root, probe_id, "pre.jpg")):
                    return root, cand_sub
                if os.path.exists(os.path.join(root, "Fenomena", probe_id, "pre.jpg")):
                    return os.path.join(root, "Fenomena"), cand_sub
    raise FileNotFoundError("Dataset Task-2 (sample_submission.csv + <Id>/pre.jpg) tidak ditemukan.")


DATA_DIR, SUB_PATH = find_task2_paths()
sub_df = pd.read_csv(SUB_PATH)
all_ids = sub_df["Id"].astype(str).tolist()
pairs_df = pd.DataFrame({
    "Id": all_ids,
    "pre_path": [os.path.join(DATA_DIR, uid, "pre.jpg") for uid in all_ids],
    "post_path": [os.path.join(DATA_DIR, uid, "post.jpg") for uid in all_ids],
})
assert all(os.path.exists(p) for p in pairs_df["pre_path"].head(5)), "Path pre.jpg contoh tidak ditemukan!"
print(f"DEVICE={DEVICE} (GPUs={torch.cuda.device_count()}) | DATA_DIR={DATA_DIR} | OUT_DIR={OUT_DIR} | N={len(pairs_df)}")

DEVICE=cuda (GPUs=2) | DATA_DIR=/kaggle/input/competitions/task-2-datavidia-11-0/Task-2/Fenomena | OUT_DIR=/kaggle/working/v3_dino_features | N=3180


In [3]:
def detect_valid_mask(rgb: np.ndarray, sum_thr: int = 10) -> np.ndarray:
    """Deteksi area piksel non-hitam valid."""
    return rgb.sum(axis=2) > sum_thr


def _extract_valid_crop(rgb: np.ndarray):
    valid_raw = detect_valid_mask(rgb)
    if int(valid_raw.sum()) < 50:
        return rgb.copy(), np.ones(rgb.shape[:2], dtype=bool), max(rgb.shape[:2]), False
    ys, xs = np.where(valid_raw)
    y0, y1 = max(0, int(ys.min()) - 4), min(rgb.shape[0], int(ys.max()) + 5)
    x0, x1 = max(0, int(xs.min()) - 4), min(rgb.shape[1], int(xs.max()) + 5)
    crop = rgb[y0:y1, x0:x1].copy()
    m_crop = valid_raw[y0:y1, x0:x1]
    side = max(crop.shape[0], crop.shape[1])
    return crop, m_crop, side, True


def _pad_inpaint_resize(crop: np.ndarray, m_crop: np.ndarray, side: int, out_size: int):
    h, w = crop.shape[:2]
    side = max(side, h, w)
    py, px = (side - h) // 2, (side - w) // 2
    sq_img = np.zeros((side, side, 3), dtype=np.uint8)
    sq_msk = np.zeros((side, side), dtype=bool)
    sq_img[py:py + h, px:px + w] = crop
    sq_msk[py:py + h, px:px + w] = m_crop

    if not sq_msk.all() and sq_msk.any():
        _, idx = distance_transform_edt(~sq_msk, return_indices=True)
        filled = sq_img.copy()
        filled[~sq_msk] = sq_img[idx[0][~sq_msk], idx[1][~sq_msk]]
        blurred = cv2.GaussianBlur(filled, (15, 15), 0)
        sq_img[~sq_msk] = blurred[~sq_msk]

    interp = cv2.INTER_AREA if side >= out_size else cv2.INTER_LINEAR
    return (
        cv2.resize(sq_img, (out_size, out_size), interpolation=interp),
        cv2.resize(sq_msk.astype(np.float32), (out_size, out_size), interpolation=cv2.INTER_AREA),
    )


def crop_pad_inpaint(rgb: np.ndarray, out_size: int, force_side: int = None):
    """Crop ketat bounding box valid, pad persegi (opsional shared side), dan inpaint tepi kosong."""
    crop, m_crop, side, ok = _extract_valid_crop(rgb)
    if not ok:
        return (
            cv2.resize(crop, (out_size, out_size), interpolation=cv2.INTER_AREA),
            np.ones((out_size, out_size), dtype=np.float32),
        )
    use_side = side if force_side is None else max(force_side, side)
    return _pad_inpaint_resize(crop, m_crop, use_side, out_size)


def crop_pad_inpaint_pair(rgb_pre: np.ndarray, rgb_post: np.ndarray, out_size: int):
    """Gunakan side_shared jika rasio skala pre/post >= 0.50 agar resolusi spasial seragam."""
    cp, mp, sp, ok_p = _extract_valid_crop(rgb_pre)
    cq, mq, sq, ok_q = _extract_valid_crop(rgb_post)
    if not ok_p or not ok_q:
        return crop_pad_inpaint(rgb_pre, out_size), crop_pad_inpaint(rgb_post, out_size), False

    ratio = min(sp, sq) / float(max(sp, sq) + 1e-6)
    if ratio >= SHARED_CROP_MIN_RATIO:
        side_shared = min(1024, max(sp, sq))
        return _pad_inpaint_resize(cp, mp, side_shared, out_size), _pad_inpaint_resize(cq, mq, side_shared, out_size), True
    return _pad_inpaint_resize(cp, mp, sp, out_size), _pad_inpaint_resize(cq, mq, sq, out_size), False


def selective_color_norm(rgb_sq: np.ndarray, msk_sq: np.ndarray, tint_thr: float = TINT_NORM_THR) -> np.ndarray:
    """Detektor tint asimetris murni (R > G AND B > G) pada skala [0, 255]; pertahankan hutan/laut alami."""
    arr = rgb_sq.astype(np.float32)
    v = msk_sq > 0.5
    if int(v.sum()) < 30:
        v = np.ones_like(msk_sq, dtype=bool)
    rm, gm, bm = arr[:, :, 0][v].mean(), arr[:, :, 1][v].mean(), arr[:, :, 2][v].mean()
    tint = max(0.0, min(float(rm - gm), float(bm - gm))) / 255.0
    if tint > tint_thr:
        mu = np.array([rm, gm, bm], dtype=np.float32)[None, None, :]
        sd = np.maximum(np.array([arr[:, :, c][v].std() for c in range(3)], dtype=np.float32), 12.0)[None, None, :]
        return np.clip(127.5 + 48.0 * ((arr - mu) / sd), 0.0, 255.0).astype(np.uint8)
    return rgb_sq


def make_gray3_clahe(rgb_sq: np.ndarray) -> np.ndarray:
    """Cabang grayscale CLAHE 3-channel yang kebal acak warna/tint."""
    g = cv2.cvtColor(rgb_sq, cv2.COLOR_RGB2GRAY)
    cg = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(g)
    return np.stack([cg, cg, cg], axis=-1)


def compute_artifact_and_phys(rgb_raw: np.ndarray) -> dict:
    """Ekstraksi fitur artifak & sinyal fisik bencana per citra (skala [0, 1] untuk pink_cast)."""
    small = cv2.resize(rgb_raw, (256, 256), interpolation=cv2.INTER_AREA)
    valid = detect_valid_mask(small)
    if int(valid.sum()) < 50:
        valid = np.ones((256, 256), dtype=bool)
    arr = small.astype(np.float32) / 255.0
    rv, gv, bv = arr[:, :, 0][valid], arr[:, :, 1][valid], arr[:, :, 2][valid]
    bright = (rv + gv + bv) / 3.0
    cmax, cmin = np.maximum(np.maximum(rv, gv), bv), np.minimum(np.minimum(rv, gv), bv)
    sat = (cmax - cmin) / (cmax + 1e-5)

    # Pure asymmetric pink_cast in [0, 1] (no /255.0!) -> 0.000 on green forests, oceans, wildfire glow, laterite soil
    pink_cast = float(max(0.0, min(float(rv.mean() - gv.mean()), float(bv.mean() - gv.mean()))))
    chroma_std = float(np.mean(np.std(arr[valid], axis=1)))
    gray_u8 = cv2.cvtColor(small, cv2.COLOR_RGB2GRAY)
    gy = np.abs(cv2.Sobel(gray_u8, cv2.CV_32F, 0, 1))
    gx = np.abs(cv2.Sobel(gray_u8, cv2.CV_32F, 1, 0))
    lap = float(cv2.Laplacian(gray_u8, cv2.CV_32F)[valid].var())

    return {
        "black_frac": float(1.0 - valid.mean()),
        "cloud_frac": float(np.mean((bright > 0.85) & (sat < 0.14))),
        "tint_score": pink_cast,
        "is_gray": float(chroma_std < 0.02),
        "chroma_std": chroma_std,
        "brightness": float(bright.mean()),
        "sat": float(sat.mean()),
        "blur_score": lap,
        "scanline_score": float((gy[valid].mean() + 1e-5) / (gx[valid].mean() + 1e-5)),
        "exg": float(np.mean(2.0 * gv - rv - bv)),
        "water": float(np.mean((bv > rv + 0.03) & (bv > gv + 0.01) & (bright < 0.65))),
        "dark_char": float(np.mean(bright < 0.22)),
        "ash_ratio": float(np.mean((sat < 0.14) & (bright >= 0.25) & (bright <= 0.72))),
        "edge": float(np.hypot(gx, gy)[valid].mean() / 255.0),
    }

In [4]:
class PatchTokenWrapper(nn.Module):
    """Ekstrak patch spasial murni dengan membuang prefix token (CLS + register tokens) secara dinamis."""
    def __init__(self, core_model, is_hf: bool, n_prefix: int):
        super().__init__()
        self.core_model = core_model
        self.is_hf = is_hf
        self.n_prefix = n_prefix

    def forward(self, x):
        if self.is_hf:
            tokens = self.core_model(pixel_values=x).last_hidden_state
        else:
            tokens = self.core_model.forward_features(x)
        return tokens[:, self.n_prefix:, :]


def load_dino_backbone():
    if BACKBONE_CHOICE == "dinov3_sat_large":
        try:
            from transformers import AutoImageProcessor, AutoModel
            repo = "facebook/dinov3-vitl16-pretrain-sat493m"
            proc = AutoImageProcessor.from_pretrained(repo, token=HF_TOKEN)
            hf_m = AutoModel.from_pretrained(repo, token=HF_TOKEN).eval().to(DEVICE)
            n_pref = 1 + int(getattr(hf_m.config, "num_register_tokens", 4))
            p_sz = int(getattr(hf_m.config, "patch_size", 16))
            mean = list(getattr(proc, "image_mean", [0.430, 0.411, 0.296]))
            std = list(getattr(proc, "image_std", [0.213, 0.156, 0.143]))
            info = {
                "model": repo, "patch_size": p_sz, "n_prefix": n_pref,
                "img_size": IMG_SIZE_V3, "dim": int(hf_m.config.hidden_size),
                "mean": mean, "std": std, "license": "DINOv3 License",
                "seed": SEED, "tint_norm_thr": TINT_NORM_THR,
                "cloud_pool_thr": CLOUD_PATCH_THR, "shared_crop_min_ratio": SHARED_CROP_MIN_RATIO,
            }
            return PatchTokenWrapper(hf_m, True, n_pref), info
        except Exception as e:
            print(f"[INFO] Fallback ke DINOv2-Large karena: {type(e).__name__}: {str(e)[:160]}")

    name = "vit_large_patch14_dinov2.lvd142m"
    m = timm.create_model(name, pretrained=True, num_classes=0, img_size=DINOV2_FALLBACK_IMG_SIZE).eval().to(DEVICE)
    n_pref = int(getattr(m, "num_prefix_tokens", 1))
    info = {
        "model": name, "patch_size": 14, "n_prefix": n_pref,
        "img_size": DINOV2_FALLBACK_IMG_SIZE, "dim": int(m.num_features),
        "mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225],
        "license": "Apache-2.0", "seed": SEED, "tint_norm_thr": TINT_NORM_THR,
        "cloud_pool_thr": CLOUD_PATCH_THR, "shared_crop_min_ratio": SHARED_CROP_MIN_RATIO,
    }
    return PatchTokenWrapper(m, False, n_pref), info


def d4_augment_batch(x: torch.Tensor, m: torch.Tensor, fl: bool, rot: int):
    """Terapkan salah satu dari 8 transformasi simetri D4 secara sinkron pada gambar & mask."""
    xf = torch.flip(x, [3]) if fl else x
    mf = torch.flip(m.float(), [2]) > 0.5 if fl else m
    xv = torch.rot90(xf, rot, [2, 3]) if rot > 0 else xf
    mv = torch.rot90(mf, rot, [1, 2]) if rot > 0 else mf
    return xv, mv


def extract_patch_grid(model, x_v: torch.Tensor, m_v: torch.Tensor):
    """Jalankan forward FP16-safe lalu kembalikan patch L2-normalized & pooled embedding ter-mask."""
    bs = x_v.shape[0]
    with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda"), dtype=torch.float16):
        toks = model(x_v)
    toks_n = F.normalize(toks.float(), dim=2)
    mw = m_v.reshape(bs, -1, 1).float()
    pooled = F.normalize((toks_n * mw).sum(dim=1) / (mw.sum(dim=1) + 1e-6), dim=1)
    return toks_n, mw, pooled


def compute_spatial_stats(cmap: np.ndarray, vmask: np.ndarray, grid_sz: int):
    """Statistik spasial perubahan patch bebas rotasi berbasis P10 + P15 + floor absolut."""
    out = {}
    keys = (
        "frac_chg_abs", "frac_chg_p10", "chg_mean_p10", "chg_p90_p10",
        "frac_chg_p15", "chg_mean_p15", "chg_p90_p15", "massive_damage_guard",
        "spatial_entropy", "top10_share", "radial_decay", "com_shift", "axis_skew",
    )
    if int(vmask.sum()) < 4:
        for k in keys:
            out[k] = 0.0
        return out, np.ones(grid_sz * grid_sz, dtype=np.float32) / float(grid_sz * grid_sz)

    vals = cmap[vmask]
    p10 = float(np.percentile(vals, 10))
    p15 = float(np.percentile(vals, 15))
    crel10 = np.maximum(cmap - p10, 0.0) * vmask.astype(np.float32)
    crel15 = np.maximum(cmap - p15, 0.0) * vmask.astype(np.float32)
    vrel10 = crel10[vmask]
    vrel15 = crel15[vmask]

    out["frac_chg_abs"] = float(np.mean(vals > 0.25))
    out["frac_chg_p10"] = float(np.mean(vrel10 > 0.12))
    out["chg_mean_p10"] = float(vrel10.mean())
    out["chg_p90_p10"] = float(np.percentile(vrel10, 90))
    out["frac_chg_p15"] = float(np.mean(vrel15 > 0.12))
    out["chg_mean_p15"] = float(vrel15.mean())
    out["chg_p90_p15"] = float(np.percentile(vrel15, 90))
    out["massive_damage_guard"] = float((out["frac_chg_p15"] < 0.05) and (out["frac_chg_abs"] > 0.50))

    # Jika kerusakan ekstrem (>85% frame) membuat P15 jatuh, pakai bobot absolut agar changed_embed tetap tajam
    if out["massive_damage_guard"] > 0.5:
        crel_use = np.maximum(cmap - 0.10, 0.0) * vmask.astype(np.float32)
    else:
        crel_use = crel15
    vrel = crel_use[vmask]
    total = float(vrel.sum()) + 1e-8
    probs = vrel / total

    out["spatial_entropy"] = float(-np.sum(probs * np.log(probs + 1e-12)) / (np.log(len(probs) + 1e-12) + 1e-8))
    k10 = max(1, int(0.10 * len(vrel)))
    out["top10_share"] = float(np.sort(vrel)[-k10:].sum() / total)

    ys, xs = np.mgrid[0:grid_sz, 0:grid_sz].astype(np.float32)
    yv, xv = ys[vmask], xs[vmask]
    cy, cx = float(np.sum(probs * yv)), float(np.sum(probs * xv))
    out["com_shift"] = float(np.hypot(cy - (grid_sz - 1) / 2.0, cx - (grid_sz - 1) / 2.0) / (grid_sz * 0.5))

    pk = int(np.argmax(crel_use))
    py, px = float(pk // grid_sz), float(pk % grid_sz)
    dist = np.hypot(yv - py, xv - px)
    out["radial_decay"] = float(-np.corrcoef(dist, vrel)[0, 1]) if dist.std() > 1e-6 and vrel.std() > 1e-6 else 0.0

    coords = np.stack([xv - cx, yv - cy], axis=1)
    cov = (coords * probs[:, None]).T @ coords
    _, eigvecs = np.linalg.eigh(cov)
    proj = coords @ eigvecs[:, -1]
    sd_p = float(np.sqrt(np.sum(probs * (proj ** 2))) + 1e-6)
    out["axis_skew"] = float(abs(np.sum(probs * (proj ** 3)) / (sd_p ** 3)))

    weights = crel_use.reshape(-1)
    wsum = float(weights.sum())
    weights = (weights / wsum) if wsum > 1e-8 else (vmask.reshape(-1).astype(np.float32) / float(vmask.sum()))
    return out, weights.astype(np.float32)

In [5]:
class PairDataset(Dataset):
    def __init__(self, df: pd.DataFrame, img_size: int, mean, std, mode: str = "color_norm"):
        self.df = df.reset_index(drop=True)
        self.img_size = img_size
        self.mode = mode
        self.mean = np.array(mean, dtype=np.float32)[:, None, None]
        self.std = np.array(std, dtype=np.float32)[:, None, None]

    def __len__(self):
        return len(self.df)

    def _to_tensor_and_cloud(self, sq_img: np.ndarray, sq_msk: np.ndarray):
        view = make_gray3_clahe(sq_img) if self.mode == "gray3" else selective_color_norm(sq_img, sq_msk)
        arr = sq_img.astype(np.float32) / 255.0
        bright = arr.mean(axis=2)
        cmax, cmin = arr.max(axis=2), arr.min(axis=2)
        sat = (cmax - cmin) / (cmax + 1e-5)
        cld = ((bright > 0.85) & (sat < 0.14)).astype(np.float32)

        chw = ((view.transpose(2, 0, 1).astype(np.float32) / 255.0) - self.mean) / self.std
        # Zero out normalized invalid border pixels so padded noise does not pollute ViT attention
        chw[:, sq_msk <= 0.5] = 0.0
        return torch.from_numpy(chw), torch.from_numpy(sq_msk), torch.from_numpy(cld)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        bgr_p, bgr_q = cv2.imread(r["pre_path"]), cv2.imread(r["post_path"])
        if bgr_p is None or bgr_q is None:
            raise FileNotFoundError(f"Gagal membaca gambar: {r['Id']}")
        (sq_p, m_p), (sq_q, m_q), _ = crop_pad_inpaint_pair(
            cv2.cvtColor(bgr_p, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(bgr_q, cv2.COLOR_BGR2RGB),
            self.img_size,
        )
        xp, mp, _ = self._to_tensor_and_cloud(sq_p, m_p)
        xq, mq, cq = self._to_tensor_and_cloud(sq_q, m_q)
        return xp, mp, xq, mq, cq


def select_cloud_excluded_mask(mq_grid: torch.Tensor, cq_grid: torch.Tensor):
    """Pool post/changed di atas patch bebas awan (<= CLOUD_PATCH_THR); fallback all_valid hanya jika <4 patch bersih."""
    vq_all = mq_grid >= 0.5
    vq_clean = vq_all & (cq_grid <= CLOUD_PATCH_THR)
    clean_cnt = vq_clean.reshape(vq_clean.shape[0], -1).sum(dim=1)
    fallback_flag = clean_cnt < 4
    vq_pool = torch.where(fallback_flag[:, None, None], vq_all, vq_clean)
    return vq_pool, vq_all, fallback_flag


def run_pairwise_extraction(model, df: pd.DataFrame, b_info: dict, mode: str = "color_norm", compute_patch_stats: bool = True):
    img_sz, p_sz, emb_dim = b_info["img_size"], b_info["patch_size"], b_info["dim"]
    g_sz = img_sz // p_sz
    ds = PairDataset(df, img_sz, b_info["mean"], b_info["std"], mode=mode)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))

    N = len(df)
    ids = df["Id"].astype(str).tolist()
    emb_pre = np.zeros((N, emb_dim), dtype=np.float32)
    emb_post = np.zeros((N, emb_dim), dtype=np.float32)
    emb_chg = np.zeros((N, emb_dim), dtype=np.float32) if compute_patch_stats else None
    pstats = []
    ptr = 0

    with torch.no_grad():
        for xp, mp, xq, mq, cq in tqdm(dl, desc=f"Extract [{mode}]"):
            bs = xp.shape[0]
            xp, mp = xp.to(DEVICE, non_blocking=True), mp.to(DEVICE, non_blocking=True)
            xq, mq, cq = xq.to(DEVICE, non_blocking=True), mq.to(DEVICE, non_blocking=True), cq.to(DEVICE, non_blocking=True)

            vp_base = F.avg_pool2d(mp.unsqueeze(1), p_sz, p_sz).squeeze(1) >= 0.5
            mq_grid = F.avg_pool2d(mq.unsqueeze(1), p_sz, p_sz).squeeze(1)
            cq_grid = F.avg_pool2d(cq.unsqueeze(1), p_sz, p_sz).squeeze(1)
            vq_pool, vq_all, fb_flag = select_cloud_excluded_mask(mq_grid, cq_grid)

            acc_pre = torch.zeros((bs, emb_dim), device=DEVICE, dtype=torch.float32)
            acc_post = torch.zeros((bs, emb_dim), device=DEVICE, dtype=torch.float32)
            pre_bank = []
            post_tok0 = None

            for fl in (False, True):
                for rot in (0, 1, 2, 3):
                    xp_v, mp_v = d4_augment_batch(xp, vp_base, fl, rot)
                    xq_v, mq_v = d4_augment_batch(xq, vq_pool, fl, rot)

                    tp_n, mp_w, gp = extract_patch_grid(model, xp_v, mp_v)
                    tq_n, _, gq = extract_patch_grid(model, xq_v, mq_v)
                    acc_pre += gp
                    acc_post += gq

                    if compute_patch_stats:
                        pre_bank.append(tp_n * mp_w)
                        if not fl and rot == 0:
                            post_tok0 = tq_n

            emb_pre[ptr:ptr + bs] = F.normalize(acc_pre, dim=1).cpu().numpy()
            emb_post[ptr:ptr + bs] = F.normalize(acc_post, dim=1).cpu().numpy()

            if compute_patch_stats:
                bank_all = torch.cat(pre_bank, dim=1)
                sim = torch.bmm(post_tok0, bank_all.transpose(1, 2))
                max_sim, _ = sim.max(dim=2)
                chg_map = (1.0 - max_sim).reshape(bs, g_sz, g_sz).cpu().numpy()
                vp_np = vq_pool.cpu().numpy()
                va_np = vq_all.cpu().numpy()
                fb_np = fb_flag.cpu().numpy()
                pt_np = post_tok0.cpu().numpy()
                for i in range(bs):
                    st, w_chg = compute_spatial_stats(chg_map[i], vp_np[i], g_sz)
                    st_all, _ = compute_spatial_stats(chg_map[i], va_np[i], g_sz)
                    st["cloud_excluded_fallback_flag"] = float(fb_np[i])
                    for k_a, v_a in st_all.items():
                        st[f"{k_a}_allvalid"] = v_a
                    st["Id"] = ids[ptr + i]
                    pstats.append(st)
                    vec = (pt_np[i] * w_chg[:, None]).sum(axis=0)
                    emb_chg[ptr + i] = vec / (np.linalg.norm(vec) + 1e-8)

            ptr += bs

    return emb_pre, emb_post, emb_chg, pd.DataFrame(pstats) if compute_patch_stats else None


# 1. Fitur Artifak & Fisik
t_start = time.time()
art_rows = []
for _, row in tqdm(pairs_df.iterrows(), total=len(pairs_df), desc="Artifak & Fisik"):
    mp = compute_artifact_and_phys(cv2.cvtColor(cv2.imread(row["pre_path"]), cv2.COLOR_BGR2RGB))
    mq = compute_artifact_and_phys(cv2.cvtColor(cv2.imread(row["post_path"]), cv2.COLOR_BGR2RGB))
    r = {"Id": row["Id"], **{f"{k}_pre": v for k, v in mp.items()}, **{f"{k}_post": v for k, v in mq.items()}}
    unrel = (mp["is_gray"] > 0.5) or (mq["is_gray"] > 0.5) or (mp["tint_score"] > TINT_NORM_THR) or (mq["tint_score"] > TINT_NORM_THR)
    r["color_unreliable_flag"] = float(unrel)
    r["delta_exg"] = mq["exg"] - mp["exg"]
    r["delta_water"] = mq["water"] - mp["water"]
    r["ash_gain"] = mq["ash_ratio"] - mp["ash_ratio"]
    r["sat_drop"] = mp["sat"] - mq["sat"]
    r["dark_char_gain"] = mq["dark_char"] - mp["dark_char"]
    r["delta_edge"] = mq["edge"] - mp["edge"]
    r["edge_ratio"] = (mq["edge"] + 1e-4) / (mp["edge"] + 1e-4)
    for ccol in ("delta_exg", "delta_water", "ash_gain", "sat_drop", "dark_char_gain"):
        r[f"{ccol}_nan"] = np.nan if unrel else r[ccol]
        r[f"{ccol}_imp0"] = 0.0 if unrel else r[ccol]
    art_rows.append(r)

art_df = pd.DataFrame(art_rows)
art_df.to_parquet(os.path.join(OUT_DIR, "artifact_and_phys_v3.parquet"), index=False)

# 2. Backbone & Ekstraksi GPU
backbone, b_info = load_dino_backbone()
if DEVICE == "cuda" and torch.cuda.device_count() > 1:
    backbone = nn.DataParallel(backbone)
print("BACKBONE:", json.dumps(b_info, indent=2))

c_pre, c_post, c_chg, patch_df = run_pairwise_extraction(backbone, pairs_df, b_info, mode="color_norm", compute_patch_stats=True)
np.save(os.path.join(OUT_DIR, "dino_color_pre.npy"), c_pre)
np.save(os.path.join(OUT_DIR, "dino_color_post.npy"), c_post)
np.save(os.path.join(OUT_DIR, "dino_changed_embed.npy"), c_chg)
patch_df.to_parquet(os.path.join(OUT_DIR, "patch_change_stats_v3.parquet"), index=False)

if EXTRACT_GRAY3:
    g_pre, g_post, _, _ = run_pairwise_extraction(backbone, pairs_df, b_info, mode="gray3", compute_patch_stats=False)
    np.save(os.path.join(OUT_DIR, "dino_gray_pre.npy"), g_pre)
    np.save(os.path.join(OUT_DIR, "dino_gray_post.npy"), g_post)

with open(os.path.join(OUT_DIR, "extract_v3_meta.json"), "w") as f:
    json.dump(b_info, f, indent=2)

# 3. Sanity Checks
assert c_pre.shape == (len(pairs_df), b_info["dim"]) and not np.isnan(c_pre).any()
assert c_post.shape == (len(pairs_df), b_info["dim"]) and not np.isnan(c_post).any()
assert c_chg.shape == (len(pairs_df), b_info["dim"]) and not np.isnan(c_chg).any()
imp0_cols = [c for c in art_df.columns if c.endswith("_imp0")]
assert art_df[imp0_cols].isna().sum().sum() == 0
print(f"SELESAI dalam {time.time() - t_start:.1f}s! Artefak tersimpan di: {OUT_DIR}")
print(f"Tint flagged pairs: {int(art_df['color_unreliable_flag'].sum())}/{len(art_df)}")

Artifak & Fisik:   0%|          | 0/3180 [00:00<?, ?it/s]

[INFO] Fallback ke DINOv2-Large karena: OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/facebook/dinov3-vitl16-pretrain-sat493m.
401 Client Error. (Requ


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

model.safetensors: downloading bytes:           |  0.00B            

BACKBONE: {
  "model": "vit_large_patch14_dinov2.lvd142m",
  "patch_size": 14,
  "n_prefix": 1,
  "img_size": 308,
  "dim": 1024,
  "mean": [
    0.485,
    0.456,
    0.406
  ],
  "std": [
    0.229,
    0.224,
    0.225
  ],
  "license": "Apache-2.0",
  "seed": 42,
  "tint_norm_thr": 0.05,
  "cloud_pool_thr": 0.5,
  "shared_crop_min_ratio": 0.5
}


Extract [color_norm]:   0%|          | 0/199 [00:00<?, ?it/s]

Extract [gray3]:   0%|          | 0/199 [00:00<?, ?it/s]

SELESAI dalam 2049.6s! Artefak tersimpan di: /kaggle/working/v3_dino_features
Tint flagged pairs: 922/3180
